# KOSDAQ VAR 기관 세분화

이 노트북은 KOSDAQ에 대해 기관을 세부 주체로 나눈 VAR 모형을 추정합니다.

- 분석기간: `2010-01-01 ~ 2025-12-31`
- 변수: `log_return`, `foreign_net_buy_ratio`, `securities_net_buy_ratio`, `insurance_net_buy_ratio`, `trust_net_buy_ratio`, `bank_net_buy_ratio`, `other_fin_net_buy_ratio`, `pension_net_buy_ratio`
- lag 탐색 범위: `1 ~ 5`
- 기본 lag 선택 기준: `BIC`
- 출력 결과: 정상성 검정, lag 선택, 안정성 확인, Granger 인과관계, IRF

기관 전체 순매수비율(`inst_net_buy_ratio`)은 세부 기관 순매수비율의 합과 중복되므로 본 노트북에서는 제외하고, 금융투자·보험·투신·은행·기타금융기관·연기금을 각각 별도 변수로 둡니다.
추가로 STEP 6에서는 환율과 금리의 일별 변동폭을 외생변수로 넣은 확장 VAR 결과도 확인합니다.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tools.sm_exceptions import ValueWarning
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


def find_kosdaq_csv() -> Path:
    candidate_paths = [
        Path("..") / "시계열 자료" / "kosdaq_clean.csv",
        Path.cwd().parent / "시계열 자료" / "kosdaq_clean.csv",
        Path("시계열 자료") / "kosdaq_clean.csv",
        Path.cwd() / "시계열 자료" / "kosdaq_clean.csv",
    ]
    for path in candidate_paths:
        resolved = path.resolve()
        if resolved.exists():
            return resolved

    matches = sorted(Path.cwd().rglob("kosdaq_clean.csv"))
    if not matches:
        raise FileNotFoundError("현재 작업 경로에서 kosdaq_clean.csv를 찾지 못했습니다.")
    return matches[0].resolve()


data_path = find_kosdaq_csv()
kosdaq_raw = pd.read_csv(data_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print(f"불러온 파일: {data_path}")
print(f"원본 기간: {kosdaq_raw['date'].min().date()} ~ {kosdaq_raw['date'].max().date()} / 행 수={len(kosdaq_raw):,}")
kosdaq_raw.head()


In [ ]:
institution_detail_map = {
    "securities_net_buy_ratio": "금융투자",
    "insurance_net_buy_ratio": "보험",
    "trust_net_buy_ratio": "투신",
    "bank_net_buy_ratio": "은행",
    "other_fin_net_buy_ratio": "기타금융기관",
    "pension_net_buy_ratio": "연기금",
}

kosdaq = kosdaq_raw.copy()
kosdaq["log_return"] = np.log(kosdaq["close_index"]).diff()

var_columns = ["log_return", "foreign_net_buy_ratio", *institution_detail_map.keys()]
analysis = kosdaq[(kosdaq["date"] >= "2010-01-01") & (kosdaq["date"] <= "2025-12-31")].copy()
var_data = analysis[["date", *var_columns]].dropna().reset_index(drop=True)
endog = var_data[var_columns].copy()

variable_info = pd.DataFrame(
    [
        {"변수명": "log_return", "설명": "KOSDAQ 로그수익률"},
        {"변수명": "foreign_net_buy_ratio", "설명": "외국인 순매수비율"},
        *[
            {"변수명": col, "설명": f"{label} 순매수비율"}
            for col, label in institution_detail_map.items()
        ],
    ]
)

print(f"VAR 분석기간: {var_data['date'].min().date()} ~ {var_data['date'].max().date()} / 행 수={len(var_data):,}")
display(variable_info)
display(var_data.head())
display(endog.describe().T.round(6))


## STEP 1. 정상성 확인

모든 VAR 변수에 대해 ADF 검정으로 정상성을 확인합니다.

In [ ]:
stationarity_records = []

for col in endog.columns:
    test_stat, p_value, used_lag, nobs, *_ = adfuller(endog[col], autolag="AIC")
    stationarity_records.append(
        {
            "변수": col,
            "설명": variable_info.set_index("변수명").loc[col, "설명"],
            "ADF 통계량": test_stat,
            "p-value": p_value,
            "사용 lag": used_lag,
            "관측치 수": nobs,
            "5% 유의수준 정상성": p_value < 0.05,
        }
    )

stationarity_table = pd.DataFrame(stationarity_records)
display(stationarity_table.round(6))


## STEP 2. lag 선택

`1`부터 `5`까지 비교한 뒤 기본적으로 `BIC` 최소값을 선택합니다. 필요하면 아래 `selected_lag_rule`을 `"AIC"`로 바꿔 비교할 수 있습니다.

In [ ]:
var_model = VAR(endog)

lag_records = []
for p_lag in range(1, 6):
    fitted = var_model.fit(p_lag)
    lag_records.append(
        {
            "lag": p_lag,
            "AIC": fitted.aic,
            "BIC": fitted.bic,
            "HQIC": fitted.hqic,
            "FPE": fitted.fpe,
        }
    )

lag_table = pd.DataFrame(lag_records).set_index("lag")
display(lag_table.round(6))

aic_lag = int(lag_table["AIC"].idxmin())
bic_lag = int(lag_table["BIC"].idxmin())
selected_lag_rule = "BIC"
selected_lag = int(lag_table[selected_lag_rule].idxmin())

print(f"AIC 기준 선택 lag: {aic_lag}")
print(f"BIC 기준 선택 lag: {bic_lag}")
print(f"최종 사용 lag: {selected_lag} ({selected_lag_rule} 기준)")


## STEP 3. VAR 추정 및 안정성 확인

선택된 lag로 VAR을 적합하고 안정성을 확인합니다. `statsmodels` 기준으로 roots modulus가 모두 `1`보다 크면 안정적입니다.

In [ ]:
var_results = var_model.fit(selected_lag)

stability_table = (
    pd.DataFrame({"root modulus": np.abs(var_results.roots)})
    .sort_values("root modulus")
    .reset_index(drop=True)
)

print(f"안정적 VAR 여부: {var_results.is_stable()}")
display(stability_table.round(6))
var_results.summary()


## STEP 4. Granger 인과관계 검정

외국인과 각 세부 기관 주체에 대해 `순매수비율 -> 수익률`, `수익률 -> 순매수비율` 방향을 각각 검정합니다.

In [ ]:
entity_label_map = {"foreign_net_buy_ratio": "외국인", **institution_detail_map}

granger_specs = [
    ("log_return", ["foreign_net_buy_ratio"], "외국인 -> 수익률"),
    ("foreign_net_buy_ratio", ["log_return"], "수익률 -> 외국인"),
]
granger_specs.extend(
    [
        ("log_return", [col], f"{label} -> 수익률")
        for col, label in institution_detail_map.items()
    ]
)
granger_specs.extend(
    [
        (col, ["log_return"], f"수익률 -> {label}")
        for col, label in institution_detail_map.items()
    ]
)

granger_records = []
for caused, causing, label in granger_specs:
    test = var_results.test_causality(caused=caused, causing=causing, kind="f")
    granger_records.append(
        {
            "검정": label,
            "종속 변수": caused,
            "종속 변수 설명": entity_label_map.get(caused, caused if caused == "log_return" else caused),
            "원인 변수": ", ".join(causing),
            "F 통계량": test.test_statistic,
            "p-value": test.pvalue,
            "자유도": str(test.df),
            "5% 유의": test.pvalue < 0.05,
        }
    )

granger_table = pd.DataFrame(granger_records)
display(granger_table.round(6))


## STEP 5. IRF

세부 기관별 shock이 수익률에 미치는 반응과 수익률 shock에 대한 세부 기관별 반응을 함께 시각화합니다. 기본 설정은 `orth=False`입니다.

In [ ]:
irf_horizon = 10
irf = var_results.irf(irf_horizon)

response_idx = {name: i for i, name in enumerate(endog.columns)}
periods = np.arange(irf_horizon + 1)
institution_cols = list(institution_detail_map.keys())
irf_impulse_label_map = {"foreign_net_buy_ratio": "외국인", **institution_detail_map}
irf_response_label_map = {
    "log_return": "KOSDAQ 수익률",
    "foreign_net_buy_ratio": "외국인",
    **institution_detail_map,
}

significant_to_return = granger_table.loc[
    (granger_table["종속 변수"] == "log_return") & (granger_table["5% 유의"]),
    "원인 변수",
].tolist()
significant_institution_to_foreign = granger_table.loc[
    (granger_table["종속 변수"] == "foreign_net_buy_ratio")
    & (granger_table["원인 변수"] != "log_return")
    & (granger_table["5% 유의"]),
    "원인 변수",
].tolist()
significant_from_return = granger_table.loc[
    (granger_table["원인 변수"] == "log_return") & (granger_table["5% 유의"]),
    "종속 변수",
].tolist()


def plot_irf_grid(response_var: str, impulse_vars: list[str], title: str, empty_message: str) -> None:
    if not impulse_vars:
        print(empty_message)
        return

    ncols = 3
    nrows = int(np.ceil(len(impulse_vars) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()

    for ax, impulse_var in zip(axes, impulse_vars):
        response_series = irf.irfs[
            :,
            response_idx[response_var],
            response_idx[impulse_var],
        ]
        ax.plot(periods, response_series, marker="o")
        ax.axhline(0, color="black", linewidth=1, alpha=0.5)
        ax.set_title(irf_impulse_label_map.get(impulse_var, impulse_var))
        ax.set_xlabel("시차")
        ax.set_ylabel("반응")
        ax.grid(alpha=0.3)

    for ax in axes[len(impulse_vars):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


def plot_return_shock_responses(response_vars: list[str], title: str, empty_message: str) -> None:
    if not response_vars:
        print(empty_message)
        return

    ncols = 3
    nrows = int(np.ceil(len(response_vars) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()

    for ax, response_var in zip(axes, response_vars):
        response_series = irf.irfs[
            :,
            response_idx[response_var],
            response_idx["log_return"],
        ]
        ax.plot(periods, response_series, marker="o")
        ax.axhline(0, color="black", linewidth=1, alpha=0.5)
        ax.set_title(f"수익률 shock -> {irf_response_label_map.get(response_var, response_var)}")
        ax.set_xlabel("시차")
        ax.set_ylabel("반응")
        ax.grid(alpha=0.3)

    for ax in axes[len(response_vars):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


print("Granger 인과관계가 5% 유의했던 방향만 IRF로 시각화합니다.")

plot_irf_grid(
    response_var="log_return",
    impulse_vars=significant_to_return,
    title="Granger 유의 관계만: KOSDAQ 수익률 반응",
    empty_message="유의한 '원인 변수 -> KOSDAQ 수익률' 관계가 없어 해당 IRF는 생략합니다.",
)

plot_irf_grid(
    response_var="foreign_net_buy_ratio",
    impulse_vars=significant_institution_to_foreign,
    title="Granger 유의 관계만: 외국인 순매수비율 반응",
    empty_message="유의한 '세부 기관 -> 외국인 순매수비율' 관계가 없어 해당 IRF는 생략합니다.",
)

plot_return_shock_responses(
    response_vars=significant_from_return,
    title="Granger 유의 관계만: KOSDAQ 수익률 shock -> 반응 변수",
    empty_message="유의한 'KOSDAQ 수익률 -> 반응 변수' 관계가 없어 해당 IRF는 생략합니다.",
)


## STEP 6. 환율·금리 변동폭을 외생변수로 넣은 확장 VAR

기본 세부기관 VAR은 그대로 두고, `usdkrw`와 `korea_3y_yield`의 일별 변동폭(1차 차분)을 외생변수로 추가한 확장 VAR도 함께 봅니다.

- 환율 변동폭: `usdkrw_change = usdkrw.diff()`
- 금리 변동폭: `korea_3y_yield_change = korea_3y_yield.diff()`
- 외생변수는 IRF의 impulse 대상이 아니라 각 내생변수 방정식의 공통 설명변수로 해석합니다.

In [ ]:
analysis_exog = analysis.copy()
analysis_exog["usdkrw_change"] = analysis_exog["usdkrw"].diff()
analysis_exog["korea_3y_yield_change"] = analysis_exog["korea_3y_yield"].diff()

varx_data = analysis_exog[
    ["date", *var_columns, "usdkrw_change", "korea_3y_yield_change"]
].dropna().reset_index(drop=True)

endog_varx = varx_data[var_columns].copy()
exog_varx = varx_data[["usdkrw_change", "korea_3y_yield_change"]].copy()

print(
    f"외생변수 포함 VAR 분석기간: "
    f"{varx_data['date'].min().date()} ~ {varx_data['date'].max().date()} / 행 수={len(varx_data):,}"
)
display(varx_data.head())
display(exog_varx.describe().T.round(6))

exog_stationarity_records = []
for col in exog_varx.columns:
    test_stat, p_value, used_lag, nobs, *_ = adfuller(exog_varx[col], autolag="AIC")
    exog_stationarity_records.append(
        {
            "변수": col,
            "ADF 통계량": test_stat,
            "p-value": p_value,
            "사용 lag": used_lag,
            "관측치 수": nobs,
            "5% 유의수준 정상성": p_value < 0.05,
        }
    )

exog_stationarity_table = pd.DataFrame(exog_stationarity_records)
display(exog_stationarity_table.round(6))


In [ ]:
varx_model = VAR(endog_varx, exog=exog_varx)

varx_lag_records = []
for p_lag in range(1, 6):
    fitted = varx_model.fit(p_lag)
    varx_lag_records.append(
        {
            "lag": p_lag,
            "AIC": fitted.aic,
            "BIC": fitted.bic,
            "HQIC": fitted.hqic,
            "FPE": fitted.fpe,
        }
    )

varx_lag_table = pd.DataFrame(varx_lag_records).set_index("lag")
display(varx_lag_table.round(6))

varx_selected_lag_rule = selected_lag_rule
varx_selected_lag = int(varx_lag_table[varx_selected_lag_rule].idxmin())

print(f"외생변수 포함 VAR의 최종 사용 lag: {varx_selected_lag} ({varx_selected_lag_rule} 기준)")


In [ ]:
varx_results = varx_model.fit(varx_selected_lag)

varx_stability_table = (
    pd.DataFrame({"root modulus": np.abs(varx_results.roots)})
    .sort_values("root modulus")
    .reset_index(drop=True)
)

print(f"안정적 VAR 여부(외생변수 포함): {varx_results.is_stable()}")
display(varx_stability_table.round(6))
varx_results.summary()


In [ ]:
varx_target_names = list(endog_varx.columns)
varx_target_label_map = {
    "log_return": "KOSDAQ 로그수익률",
    "foreign_net_buy_ratio": "외국인 순매수비율",
    **{col: f"{label} 순매수비율" for col, label in institution_detail_map.items()},
}
varx_param_names = ["const", *list(exog_varx.columns)] + [
    f"L{lag}.{name}"
    for lag in range(1, varx_selected_lag + 1)
    for name in varx_target_names
]

varx_raw_params = pd.DataFrame(
    np.asarray(varx_results._results.params),
    index=varx_param_names,
    columns=varx_target_names,
)
varx_raw_pvalues = pd.DataFrame(
    np.asarray(varx_results._results.pvalues),
    index=varx_param_names,
    columns=varx_target_names,
)

exog_effect_records = []
for exog_name in exog_varx.columns:
    for target in varx_target_names:
        exog_effect_records.append(
            {
                "외생 변수": exog_name,
                "종속 변수": target,
                "종속 변수 설명": varx_target_label_map[target],
                "계수": varx_raw_params.loc[exog_name, target],
                "p-value": varx_raw_pvalues.loc[exog_name, target],
                "5% 유의": varx_raw_pvalues.loc[exog_name, target] < 0.05,
            }
        )

exog_effect_table = pd.DataFrame(exog_effect_records)
model_comparison_table = pd.DataFrame(
    [
        {
            "모형": "기본 VAR",
            "관측치 수": len(var_data),
            "lag": selected_lag,
            "AIC": var_results.aic,
            "BIC": var_results.bic,
            "안정적": var_results.is_stable(),
        },
        {
            "모형": "외생변수 포함 VAR",
            "관측치 수": len(varx_data),
            "lag": varx_selected_lag,
            "AIC": varx_results.aic,
            "BIC": varx_results.bic,
            "안정적": varx_results.is_stable(),
        },
    ]
)

print("외생변수 계수 요약")
display(exog_effect_table.round(6))

print("기본 VAR과 외생변수 포함 VAR 비교")
display(model_comparison_table.round(6))

print("5% 유의한 외생변수 효과")
display(exog_effect_table[exog_effect_table["5% 유의"]].round(6))


## STEP 7. 외생변수 포함 VAR의 Granger 인과관계와 IRF

확장 VAR(`varx_results`)에 대해서도 내생변수간 Granger 인과관계와 IRF를 확인합니다. 환율·금리 변화폭은 외생변수이므로 impulse 대상에서는 제외하고, 내생변수 반응을 중심으로 해석합니다.


In [ ]:

varx_entity_label_map = {
    "log_return": "KOSDAQ 로그수익률",
    "foreign_net_buy_ratio": "외국인 순매수비율",
    **{col: f"{label} 순매수비율" for col, label in institution_detail_map.items()},
}

varx_granger_specs = [
    ("log_return", ["foreign_net_buy_ratio"], "외국인 -> 수익률"),
    ("foreign_net_buy_ratio", ["log_return"], "수익률 -> 외국인"),
]
varx_granger_specs.extend(
    [
        ("log_return", [col], f"{label} -> 수익률")
        for col, label in institution_detail_map.items()
    ]
)
varx_granger_specs.extend(
    [
        (col, ["log_return"], f"수익률 -> {label}")
        for col, label in institution_detail_map.items()
    ]
)

varx_granger_records = []
for caused, causing, label in varx_granger_specs:
    test = varx_results.test_causality(caused=caused, causing=causing, kind="f")
    varx_granger_records.append(
        {
            "가설": label,
            "종속 변수": caused,
            "종속 변수 설명": varx_entity_label_map.get(caused, caused),
            "원인 변수": ", ".join(causing),
            "F 통계량": test.test_statistic,
            "p-value": test.pvalue,
            "자유도": str(test.df),
            "5% 유의": test.pvalue < 0.05,
        }
    )

varx_granger_table = pd.DataFrame(varx_granger_records)
print("외생변수 포함 VAR의 Granger 인과관계")
display(varx_granger_table.round(6))


In [ ]:

varx_irf_horizon = irf_horizon
varx_irf = varx_results.irf(varx_irf_horizon)

varx_response_idx = {name: i for i, name in enumerate(endog_varx.columns)}
varx_periods = np.arange(varx_irf_horizon + 1)
varx_institution_cols = list(institution_detail_map.keys())
varx_irf_impulse_label_map = {"foreign_net_buy_ratio": "외국인", **institution_detail_map}
varx_irf_response_label_map = {
    "log_return": "KOSDAQ 수익률",
    "foreign_net_buy_ratio": "외국인",
    **institution_detail_map,
}

varx_significant_to_return = varx_granger_table.loc[
    (varx_granger_table["종속 변수"] == "log_return") & (varx_granger_table["5% 유의"]),
    "원인 변수",
].tolist()
varx_significant_institution_to_foreign = varx_granger_table.loc[
    (varx_granger_table["종속 변수"] == "foreign_net_buy_ratio")
    & (varx_granger_table["원인 변수"] != "log_return")
    & (varx_granger_table["5% 유의"]),
    "원인 변수",
].tolist()
varx_significant_from_return = varx_granger_table.loc[
    (varx_granger_table["원인 변수"] == "log_return") & (varx_granger_table["5% 유의"]),
    "종속 변수",
].tolist()


def plot_varx_irf_grid(response_var: str, impulse_vars: list[str], title: str, empty_message: str) -> None:
    if not impulse_vars:
        print(empty_message)
        return

    ncols = 3
    nrows = int(np.ceil(len(impulse_vars) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()

    for ax, impulse_var in zip(axes, impulse_vars):
        response_series = varx_irf.irfs[
            :,
            varx_response_idx[response_var],
            varx_response_idx[impulse_var],
        ]
        ax.plot(varx_periods, response_series, marker="o")
        ax.axhline(0, color="black", linewidth=1, alpha=0.5)
        ax.set_title(varx_irf_impulse_label_map.get(impulse_var, impulse_var))
        ax.set_xlabel("시차")
        ax.set_ylabel("반응")
        ax.grid(alpha=0.3)

    for ax in axes[len(impulse_vars):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


def plot_varx_return_shock_responses(response_vars: list[str], title: str, empty_message: str) -> None:
    if not response_vars:
        print(empty_message)
        return

    ncols = 3
    nrows = int(np.ceil(len(response_vars) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharex=True)
    axes = np.atleast_1d(axes).flatten()

    for ax, response_var in zip(axes, response_vars):
        response_series = varx_irf.irfs[
            :,
            varx_response_idx[response_var],
            varx_response_idx["log_return"],
        ]
        ax.plot(varx_periods, response_series, marker="o")
        ax.axhline(0, color="black", linewidth=1, alpha=0.5)
        ax.set_title(f"수익률 shock -> {varx_irf_response_label_map.get(response_var, response_var)}")
        ax.set_xlabel("시차")
        ax.set_ylabel("반응")
        ax.grid(alpha=0.3)

    for ax in axes[len(response_vars):]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()


print("외생변수 포함 VAR도 Granger 인과관계가 5% 유의했던 방향만 IRF로 시각화합니다.")

plot_varx_irf_grid(
    response_var="log_return",
    impulse_vars=varx_significant_to_return,
    title="외생변수 포함 VAR: Granger 유의 관계만 KOSDAQ 수익률 반응",
    empty_message="외생변수 포함 VAR에서 유의한 '원인 변수 -> KOSDAQ 수익률' 관계가 없어 해당 IRF는 생략합니다.",
)

plot_varx_irf_grid(
    response_var="foreign_net_buy_ratio",
    impulse_vars=varx_significant_institution_to_foreign,
    title="외생변수 포함 VAR: Granger 유의 관계만 외국인 순매수비율 반응",
    empty_message="외생변수 포함 VAR에서 유의한 '세부 기관 -> 외국인 순매수비율' 관계가 없어 해당 IRF는 생략합니다.",
)

plot_varx_return_shock_responses(
    response_vars=varx_significant_from_return,
    title="외생변수 포함 VAR: Granger 유의 관계만 KOSDAQ 수익률 shock -> 반응 변수",
    empty_message="외생변수 포함 VAR에서 유의한 'KOSDAQ 수익률 -> 반응 변수' 관계가 없어 해당 IRF는 생략합니다.",
)
